# Project 1: Missing Metadata by Event Date

This notebook extracts rows missing any of four metadata fields from the assignment dataset, splits them at the median `EventDate` into older and newer groups, sorts each group chronologically, and saves each group separately.

In [ ]:
from collections import Counter
from pathlib import Path
import pandas as pd

input_path = "Data_set_prep_assignment_1.csv"
temporary_path = Path("project_1_missing_metadata_rows_unsorted.tmp.csv")
older_output_path = "project_1_missing_metadata_older_by_event_date.csv"
newer_output_path = "project_1_missing_metadata_newer_by_event_date.csv"
metadata_fields = [
    "DaysInMilk",
    "AnimalId",
    "LactationNumber",
    "ReproductionStatus",
]


In [ ]:
if temporary_path.exists():
    temporary_path.unlink()

date_counts = Counter()
total_source_rows = 0
total_missing_metadata_rows = 0
total_missing_all_metadata = 0
write_header = True

for chunk in pd.read_csv(input_path, chunksize=300_000, low_memory=False):
    total_source_rows += len(chunk)
    missing_any_metadata = chunk[metadata_fields].isna().any(axis=1)
    total_missing_all_metadata += int(chunk[metadata_fields].isna().all(axis=1).sum())
    selected_rows = chunk.loc[missing_any_metadata].copy()
    total_missing_metadata_rows += len(selected_rows)

    if selected_rows.empty:
        continue

    selected_rows["EventDate"] = pd.to_datetime(
        selected_rows["EventDate"], errors="coerce"
    )
    if selected_rows["EventDate"].isna().any():
        raise ValueError("Some selected rows have missing or invalid EventDate values.")

    date_counts.update(
        selected_rows["EventDate"].dt.strftime("%Y-%m-%d").value_counts().to_dict()
    )
    selected_rows.to_csv(
        temporary_path,
        mode="w" if write_header else "a",
        header=write_header,
        index=False,
    )
    write_header = False

if not date_counts:
    raise ValueError("No rows with missing metadata were found.")

# Use the lower median date as the split point; dates on the cutoff go in newer.
median_position = (sum(date_counts.values()) - 1) // 2
cumulative_rows = 0
split_date = None
for date_value, date_count in sorted(date_counts.items()):
    cumulative_rows += date_count
    if cumulative_rows > median_position:
        split_date = pd.Timestamp(date_value)
        break

missing_metadata_rows = pd.read_csv(
    temporary_path, parse_dates=["EventDate"], low_memory=False
)
missing_metadata_rows.sort_values("EventDate", kind="mergesort", inplace=True)
older_rows = missing_metadata_rows.loc[
    missing_metadata_rows["EventDate"] < split_date
]
newer_rows = missing_metadata_rows.loc[
    missing_metadata_rows["EventDate"] >= split_date
]

older_rows.to_csv(older_output_path, index=False)
newer_rows.to_csv(newer_output_path, index=False)

if len(older_rows) + len(newer_rows) != total_missing_metadata_rows:
    raise AssertionError("The date split did not preserve every selected row.")

temporary_path.unlink()

print(f"Source rows: {total_source_rows:,}")
print(f"Rows missing any of the four metadata fields: {total_missing_metadata_rows:,}")
print(f"Rows missing all four metadata fields: {total_missing_all_metadata:,}")
print(f"Median EventDate cutoff: {split_date.date()}")
print(f"Older rows (EventDate before cutoff): {len(older_rows):,}")
print(f"Newer rows (EventDate on or after cutoff): {len(newer_rows):,}")
print(f"Saved older group to {older_output_path}")
print(f"Saved newer group to {newer_output_path}")

display(older_rows.head(), newer_rows.head())